<a href="https://colab.research.google.com/github/xwx-codingweb/XIA_DSPN_S26/blob/master/ExerciseSubmissions/15_power-analysis-via-simulations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 15: Power analyses

This  assignment is designed to give you practice with Monte Carlo methods to conduct power analyses via simulation. You won't need to load in any data for this homework. We will, however, be using parts of the homework from last week.

---
## 1. Simulating data (1 points)


Pull your `simulate_data()` function from your last homework and add it below.

As a reminder, this function simulates the relationship between age, word reading experience, and reading comprehension skill.

`c` is reading comprehension, and `x` is word reading experience.

In [4]:
sample_size = 100 # How many children in data set?
age_lo = 80     # minimum age, in months
age_hi = 200    # maximum age, in months
beta_xa = 0.5   # amount by which experience changes for increase of one month in age
beta_x0 = -5    # amount of experience when age = 0 (not interpretable, since minimum age for this data is 80 months)
sd_x = 50       # standard dev of gaussian noise term, epsilon_x
beta_ca = 0.8   # amount that comprehension score improves for every increase of one unit in age
beta_cx = 3     # amount that comprehension score improves for every increase of one unit in reading experience
beta_c0 = 10    # comprehension score when reading experience is 0.
sd_c = 85      # standard dev of gaussian noise term, epsilon_c

simulate_data <- function(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {
      # WRITE YOUR CODE HERE
      # age
      age <- runif(sample_size, min = age_lo, max = age_hi)

      # noise
      eps_x <- rnorm(sample_size, 0, sd_x)
      eps_c <- rnorm(sample_size, 0, sd_c)

      # mediator
      x <- beta_xa * age + beta_x0 + eps_x

      # outcome
      c <- beta_ca * age + beta_cx * x + beta_c0 + eps_c

      return(data.frame(age = age, x = x, c = c))
}

dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)
head(dat)

,age,x,c
,<dbl>,<dbl>,<dbl>
1,94.42860,-17.99493,169.2630
2,98.41667,56.59382,396.1699
3,129.70413,74.88042,322.2903
4,133.12207,38.47052,236.9526
5,194.77838,86.55175,438.0237
6,86.51700,27.65196,142.0595


---
## 2. `run_analysis()` function (2 pts)

Last week, we looked at whether word reading experience(`x`) mediated the relation between `age` and reading comprehension (`c`).

Now we're going to use our `simulate_data()` function to conduct a power analysis. The goal is to determine how many participants we would need in order to detect both the mediated and the direct effects in this data.

*Note: We're going to pretend for the sake of simplicity that we don't have any control over the ages of the children we get (so ages are generated using `runif(sample_size, age_lo, age_hi)`, although of course this would be an unusual situation in reality.*

First, write a function, `run_analysis()`, that takes in simulated data, runs **your mediation from last week**, and returns a vector containing the ACME and ADE estimates and p-values (these are the `d0`, `d0.p`, `z0`, and `z0.p` features of the mediated model object, e.g., `fitMed$d0.p`). Print this function's output for the data we simulated previously.

In [6]:
# WRITE YOUR CODE HERE
install.packages("mediation")

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘checkmate’, ‘rbibutils’, ‘zoo’, ‘gridExtra’, ‘htmlTable’, ‘colorspace’, ‘Formula’, ‘Rdpack’, ‘minqa’, ‘nloptr’, ‘reformulas’, ‘RcppEigen’, ‘mvtnorm’, ‘sandwich’, ‘lpSolve’, ‘Hmisc’, ‘lme4’


Loading required package: MASS

Loading required package: Matrix

Loading required package: mvtnorm

Loading required package: sandwich

mediation: Causal Mediation Analysis
Version: 4.5.1




In [7]:
library(mediation)
run_analysis <- function(dat) {

  # mediator model
  med_model <- lm(x ~ age, data = dat)

  # outcome model
  out_model <- lm(c ~ age + x, data = dat)

  # mediation
  fitMed <- mediate(med_model, out_model,
                    treat = "age",
                    mediator = "x",
                    boot = TRUE)

  # return ACME + ADE estimates + p-values
  return(c(fitMed$d0, fitMed$d0.p,
           fitMed$z0, fitMed$z0.p))
}

dat <- simulate_data(sample_size, age_lo, age_hi, beta_xa,
                     beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c)

run_analysis(dat)

Running nonparametric bootstrap




[1] 1.141064 0.004000 1.036126 0.000000

---
## 3. `repeat_analysis()` function (3 pts)

Next fill in the function `repeat_analysis()` below so that it simulates and analyzes data `num_simulations` times. Store the outputs from each simulation in the `simouts` matrix. Calculate and return the coverage across all the simulations run for both ACME and ADE.

In [9]:
repeat_analysis <- function(num_simulations, alpha, sample_size, age_lo, age_hi,
        beta_xa, beta_x0, sd_x, beta_ca, beta_cx, beta_c0, sd_c) {

  simouts <- matrix(NA, nrow = num_simulations, ncol = 4)

  for (i in 1:num_simulations) {

    dat <- simulate_data(sample_size, age_lo, age_hi,
                         beta_xa, beta_x0, sd_x,
                         beta_ca, beta_cx, beta_c0, sd_c)

    simouts[i, ] <- run_analysis(dat)
  }

  # coverage = proportion of p-values < alpha
  ACME_cov <- mean(simouts[,2] < alpha)
  ADE_cov  <- mean(simouts[,4] < alpha)

  return(list(ACME_cov = ACME_cov, ADE_cov = ADE_cov))
}

Now run the `repeat_analysis()` function using the same parameter settings as above, for 10 simulations, with an alpha criterion of 0.01.

In [10]:
# WRITE YOUR CODE HERE
set.seed(13)

repeat_analysis(10, 0.01, sample_size,
                age_lo, age_hi,
                beta_xa, beta_x0, sd_x,
                beta_ca, beta_cx, beta_c0, sd_c)


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap




$ACME_cov
[1] 0.8

$ADE_cov
[1] 0.4

---
## 4. Testing different sample sizes (2 pts)

Finally, do the same thing (10 simulations, alpha criterion of 0.01) but for 5 different sample sizes: 50, 75, 100, 125, 150. You can do this using `map` (as in the tutorial), or a simple `for` loop, or by calculating each individually. Up to you! This should take around 3 minutes to run.

In [11]:
# WRITE YOUR CODE HERE
sample_sizes <- c(50, 75, 100, 125, 150)

results <- lapply(sample_sizes, function(n) {
  repeat_analysis(10, 0.01, n,
                  age_lo, age_hi,
                  beta_xa, beta_x0, sd_x,
                  beta_ca, beta_cx, beta_c0, sd_c)
})

names(results) <- sample_sizes

Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonparametric bootstrap


Running nonpar

Print your results.

In [12]:
# WRITE YOUR CODE HERE
results

$`50`
$`50`$ACME_cov
[1] 0.4

$`50`$ADE_cov
[1] 0.5


$`75`
$`75`$ACME_cov
[1] 0.6

$`75`$ADE_cov
[1] 0.3


$`100`
$`100`$ACME_cov
[1] 1

$`100`$ADE_cov
[1] 0.5


$`125`
$`125`$ACME_cov
[1] 1

$`125`$ADE_cov
[1] 0.7


$`150`
$`150`$ACME_cov
[1] 0.8

$`150`$ADE_cov
[1] 0.8

## 5. Reflection (2 pts)

If this were a real power analysis, we'd want to run more simulations per sample size (to get a more precise estimate of power) and we may also want to test out some other values of the parameters we used to simulate our data. However, what would you conclude just based on the results above?

> The results show that power increases with sample size for both ACME and ADE. However, the mediated effect (ACME) is detected more consistently across simulations, while the direct effect (ADE) requires larger sample sizes to achieve similar power.

**Given** how we generated the data, why was the direct effect harder to detect than the mediated effect?
> The direct effect is harder to detect because much of the effect of age on comprehension is transmitted through the mediator (reading experience). This reduces the remaining direct effect size, making it smaller and harder to detect relative to the indirect effect.

**DUE:** 5pm EST, March 31, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*